<a href="https://colab.research.google.com/github/dJasawat/ByNethaji_DeepLearing_Notebooks/blob/main/module24_experiment2_flan_t5_dialogsum_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 24 — Experiment 2  
## Real Hugging Face Fine-Tuning: FLAN‑T5 + DialogSum Conversation Summarizer

This is a second, more realistic Module 24 experiment.

### Story

In Module 23, our agent could use tools and prepare a report.

Now imagine a support operations team receives long messy conversations every day.

A manager does not want the full dialogue.  
They want a short, clean handover summary:

```text
What happened?
Who needs action?
What is the customer issue?
What should the next agent or manager know?
```

So in this experiment, students fine-tune a real Hugging Face model to convert **dialogues → summaries**.

### Main model

```text
google/flan-t5-small
```

### Main dataset

```text
knkarthick/dialogsum
```

### What makes this different from the first Module 24 notebook?

The first notebook was a very stable PyTorch-only concept lab.  
This notebook uses the Hugging Face ecosystem directly:

```text
Hugging Face Dataset → Hugging Face Tokenizer → Hugging Face Seq2Seq Model → Fine-tuning → Evaluation → Optimization thinking
```

This is closer to a real-world workflow, while still keeping the code simple and classroom-friendly.

# Curriculum coverage map

| Module 24 topic | Covered here? | How |
|---|---|---|
| Introduction to fine-tuning LLMs | Yes | Dialogue summarization story |
| Fine-tuning vs prompting | Yes | Baseline model vs fine-tuned model |
| SFT dataset format | Yes | Instruction + dialogue + summary |
| Hugging Face dataset | Yes | `knkarthick/dialogsum` |
| Hugging Face model | Yes | `google/flan-t5-small` |
| Full fine-tuning | Yes | Simple PyTorch training loop updates all trainable weights |
| PEFT / LoRA | Yes | Concept + optional guarded PEFT cell |
| LLaMA / Mistral / larger model pattern | Yes | Roadmap section |
| Evaluation | Yes | Baseline vs fine-tuned vs reference |
| Generalization | Yes | Unseen dialogue tests |
| Quantization | Yes | INT8 dynamic quantization attempt + fallback explanation |
| 8-bit vs 4-bit | Yes | Decision table |
| PTQ vs QAT | Yes | Concept section |
| ONNX / OpenVINO / Intel Neural Compressor | Yes | Production tool map |
| Safe and ethical deployment | Yes | Hallucination, privacy, and PII discussion |
| Bridge to Module 25 | Yes | Preference comparison: Summary A vs Summary B |

# Recommended 4-hour teaching flow

| Time | Section | Teaching goal | Live activity |
|---:|---|---|---|
| 0:00–0:20 | Story + model strategy | Why summarization fine-tuning is useful | Show messy dialogue |
| 0:20–0:45 | Load Hugging Face dataset | Real dataset workflow | Inspect DialogSum |
| 0:45–1:10 | Load FLAN‑T5 baseline | Prompt-only baseline | Generate summary before fine-tuning |
| 1:10–1:45 | SFT formatting | Convert data into training examples | Build instruction-format examples |
| 1:45–2:25 | Full fine-tuning | Update model behaviour | Run mini training |
| 2:25–2:55 | Evaluation | Baseline vs fine-tuned vs reference | Compare outputs |
| 2:55–3:20 | LoRA/PEFT | Why adapter training matters | Parameter-count discussion |
| 3:20–3:40 | Quantization | Smaller/faster deployment | INT8 demo or fallback explanation |
| 3:40–4:00 | Module 25 bridge | Human preference data | Choose better summary |

# Important stability design

This notebook uses a real Hugging Face model and dataset, so it needs internet access in Colab.

To keep it simple:

- training uses a small subset of the dataset,
- training uses a simple PyTorch loop instead of a complex Trainer setup,
- ROUGE is not mandatory,
- PEFT/LoRA is optional and guarded,
- quantization is attempted safely with `try/except`,
- heavy LLaMA/Mistral/4-bit training is explained as a production extension, not forced live.

### Recommended Colab setup

```text
Runtime → Restart runtime
Runtime → Change runtime type → GPU if available
Run from top
```

CPU can still work with very small training steps, but GPU is better.

# Part 1 — Install packages

We install only the core Hugging Face packages needed for this experiment.

`transformers` gives us the FLAN‑T5 model and tokenizer.  
`datasets` gives us the DialogSum dataset.  
`sentencepiece` helps with T5 tokenization.  
`accelerate` is useful for Hugging Face/PyTorch device compatibility.

In [ ]:
import sys
import subprocess

packages = [
    "transformers>=4.45.0",
    "datasets>=3.0.0",
    "accelerate>=0.33.0",
    "sentencepiece>=0.2.0"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("Installed core packages.")

Installed core packages.


# Part 2 — Imports and configuration

The notebook is intentionally small.

You can control runtime pressure using:

```python
TRAIN_SIZE
VAL_SIZE
TRAINING_STEPS
```

For a first classroom run, keep these small.

In [ ]:
from pathlib import Path
import json
import time
import random
import re
import math
import os
import tempfile
from typing import List, Dict, Any

import torch
import torch.nn as nn

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("module24_flan_t5_dialogsum_project")
PROJECT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "google/flan-t5-small"
DATASET_NAME = "knkarthick/dialogsum"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Keep these small for classroom stability.
TRAIN_SIZE = 80 if device == "cuda" else 24
VAL_SIZE = 12 if device == "cuda" else 6
TRAINING_STEPS = 40 if device == "cuda" else 10
BATCH_SIZE = 2 if device == "cuda" else 1

MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 96

print("Device:", device)
print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("TRAIN_SIZE:", TRAIN_SIZE)
print("VAL_SIZE:", VAL_SIZE)
print("TRAINING_STEPS:", TRAINING_STEPS)

Device: cpu
Model: google/flan-t5-small
Dataset: knkarthick/dialogsum
TRAIN_SIZE: 24
VAL_SIZE: 6
TRAINING_STEPS: 10


# Part 3 — Load the Hugging Face dataset

We use DialogSum.

Each row contains:

| Column | Meaning |
|---|---|
| `dialogue` | multi-turn conversation |
| `summary` | human-written summary |
| `topic` | short topic label |

For Module 24, the important learning point is:

```text
Fine-tuning data is usually pairs of input and desired output.
```

Here:

```text
Input  = dialogue
Output = summary
```

In [ ]:
raw_dataset = load_dataset(DATASET_NAME)

print(raw_dataset)
print("\nAvailable splits:", list(raw_dataset.keys()))
print("\nColumns in train split:", raw_dataset["train"].column_names)

sample = raw_dataset["train"][0]
print("\nSample keys:", sample.keys())
print("\nDIALOGUE:\n", sample["dialogue"][:800])
print("\nREFERENCE SUMMARY:\n", sample["summary"])
print("\nTOPIC:", sample.get("topic", "N/A"))

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

Available splits: ['train', 'validation', 'test']

Columns in train split: ['id', 'dialogue', 'summary', 'topic']

Sample keys: dict_keys(['id', 'dialogue', 'summary', 'topic'])

DIALOGUE:
 #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: 

# Part 4 — Why this is an SFT task

SFT means **Supervised Fine-Tuning**.

We show the model examples like:

```text
Instruction:
Summarize this dialogue for a manager.

Input:
<Person1> ...
<Person2> ...

Response:
Human-written summary
```

That is exactly the same pattern used in many instruction-tuning workflows:

```text
instruction + input → ideal response
```

In [ ]:
def make_instruction(dialogue: str) -> str:
    return (
        "Summarize this dialogue for a support operations manager.\n"
        "Write a short, clear handover summary.\n\n"
        f"Dialogue:\n{dialogue}\n\n"
        "Summary:"
    )

def make_sft_record(row: Dict[str, Any]) -> Dict[str, str]:
    return {
        "instruction": "Summarize this dialogue for a support operations manager.",
        "input": row["dialogue"],
        "response": row["summary"],
        "prompt": make_instruction(row["dialogue"])
    }

sft_example = make_sft_record(raw_dataset["train"][0])

print("INSTRUCTION:")
print(sft_example["instruction"])
print("\nINPUT PREVIEW:")
print(sft_example["input"][:600])
print("\nRESPONSE:")
print(sft_example["response"])

INSTRUCTION:
Summarize this dialogue for a support operations manager.

INPUT PREVIEW:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1

RESPONSE:
Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.


# Part 5 — Save a few SFT examples as JSONL

This is useful because many real fine-tuning pipelines expect JSONL-style records.

Each line is one training example.

In [ ]:
jsonl_path = PROJECT_DIR / "dialogsum_sft_examples.jsonl"

with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in raw_dataset["train"].select(range(10)):
        f.write(json.dumps(make_sft_record(row), ensure_ascii=False) + "\n")

print("Saved:", jsonl_path)
print("\nFirst two JSONL records:")
for i, line in enumerate(jsonl_path.read_text(encoding="utf-8").splitlines()[:2], start=1):
    print(f"\n--- record {i} ---")
    print(line[:900])

Saved: module24_flan_t5_dialogsum_project/dialogsum_sft_examples.jsonl

First two JSONL records:

--- record 1 ---
{"instruction": "Summarize this dialogue for a support operations manager.", "input": "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Perso

# Part 6 — Load the baseline model

We load `google/flan-t5-small`.

This is already instruction-tuned, so it can summarize before fine-tuning.

The teaching question is:

```text
Can fine-tuning make it behave more consistently for our specific summarization style?
```

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def count_parameters(model) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable}

params = count_parameters(model)
print("Total parameters:", f"{params['total']:,}")
print("Trainable parameters:", f"{params['trainable']:,}")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total parameters: 76,961,152
Trainable parameters: 76,961,152


# Part 7 — Baseline generation before fine-tuning

We first ask the original model to summarize a dialogue.

This is the **prompt-only baseline**.

Later, we compare:

```text
Baseline summary
vs
Fine-tuned summary
vs
Human reference summary
```

In [ ]:
@torch.no_grad()
def generate_summary(model, dialogue: str, max_new_tokens: int = 80) -> str:
    model.eval()
    prompt = make_instruction(dialogue)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=2,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

demo_row = raw_dataset["validation"][0]
baseline_summary = generate_summary(model, demo_row["dialogue"])

print("DIALOGUE PREVIEW:")
print(demo_row["dialogue"][:900])
print("\nBASELINE MODEL SUMMARY:")
print(baseline_summary)
print("\nREFERENCE HUMAN SUMMARY:")
print(demo_row["summary"])

DIALOGUE PREVIEW:
#Person1#: Hello, how are you doing today?
#Person2#: I ' Ve been having trouble breathing lately.
#Person1#: Have you had any type of cold lately?
#Person2#: No, I haven ' t had a cold. I just have a heavy feeling in my chest when I try to breathe.
#Person1#: Do you have any allergies that you know of?
#Person2#: No, I don ' t have any allergies that I know of.
#Person1#: Does this happen all the time or mostly when you are active?
#Person2#: It happens a lot when I work out.
#Person1#: I am going to send you to a pulmonary specialist who can run tests on you for asthma.
#Person2#: Thank you for your help, doctor.

BASELINE MODEL SUMMARY:
The doctor is going to send you to a pulmonary specialist who can run tests on you.

REFERENCE HUMAN SUMMARY:
#Person2# has trouble breathing. The doctor asks #Person2# about it and will send #Person2# to a pulmonary specialist.


# Part 8 — Prepare a small training subset

We do not train on the full dataset during class.

For teaching, small subsets are enough to show the full process:

```text
load data → tokenize → train → evaluate → compare
```

Production training would use much more data, proper validation, logging, and monitoring.

In [ ]:
train_rows = list(raw_dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE)))
val_rows = list(raw_dataset["validation"].shuffle(seed=SEED).select(range(VAL_SIZE)))

print("Training rows:", len(train_rows))
print("Validation rows:", len(val_rows))

print("\nExample training row:")
print("Dialogue:", train_rows[0]["dialogue"][:500])
print("\nSummary:", train_rows[0]["summary"])

Training rows: 24
Validation rows: 6

Example training row:
Dialogue: #Person1#: Hello, Anna speaking!
#Person2#: Hey, Anna, this is Jason.
#Person1#: Jason, where have you been hiding lately? You know it's been a long time since your last call. Have you been good?
#Person2#: Yes. How are you, Anna?
#Person1#: I am fine. What have you been doing?
#Person2#: Working. I've been really busy these days. I got a promotion.
#Person1#: That's great, congratulations!
#Person2#: Thanks. I am feeling pretty good about myself too. You know, bigger office, a raise and even an

Summary: Jason hasn't called Anna for a long time. He calls her to tell her he got a promotion and he feels good about it. Anna invites him to come over to her house tonight to get drunk.


# Part 9 — Tokenization

Seq2Seq models like T5 need:

- `input_ids` for the prompt/dialogue,
- `attention_mask`,
- `labels` for the target summary.

Padding tokens in labels are replaced by `-100` so PyTorch ignores them in the loss.

In [ ]:
def tokenize_row(row: Dict[str, Any]) -> Dict[str, Any]:
    prompt = make_instruction(row["dialogue"])

    model_inputs = tokenizer(
        prompt,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    labels = tokenizer(
        row["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )["input_ids"][0]

    labels = [
        token_id if token_id != tokenizer.pad_token_id else -100
        for token_id in labels.tolist()
    ]

    return {
        "input_ids": model_inputs["input_ids"][0].tolist(),
        "attention_mask": model_inputs["attention_mask"][0].tolist(),
        "labels": labels
    }

tokenized_train = [tokenize_row(row) for row in train_rows]
tokenized_val = [tokenize_row(row) for row in val_rows]

print("Tokenized training examples:", len(tokenized_train))
print("Length of input_ids:", len(tokenized_train[0]["input_ids"]))
print("Length of labels:", len(tokenized_train[0]["labels"]))

Tokenized training examples: 24
Length of input_ids: 384
Length of labels: 96


# Part 10 — A tiny batching function

This keeps training transparent.

Instead of hiding everything inside a high-level Trainer, students can see the actual objects sent to the model.

In [ ]:
def make_batch(tokenized_rows: List[Dict[str, Any]], batch_size: int) -> Dict[str, torch.Tensor]:
    rows = random.sample(tokenized_rows, k=batch_size)
    batch = {
        "input_ids": torch.tensor([r["input_ids"] for r in rows], dtype=torch.long).to(device),
        "attention_mask": torch.tensor([r["attention_mask"] for r in rows], dtype=torch.long).to(device),
        "labels": torch.tensor([r["labels"] for r in rows], dtype=torch.long).to(device)
    }
    return batch

test_batch = make_batch(tokenized_train, BATCH_SIZE)

for key, value in test_batch.items():
    print(key, value.shape)

input_ids torch.Size([1, 384])
attention_mask torch.Size([1, 384])
labels torch.Size([1, 96])


# Part 11 — Full fine-tuning

This is **full fine-tuning** because all model parameters remain trainable.

In a large model, this can be expensive.  
Here we use FLAN‑T5-small and a tiny number of steps so students can see the idea safely.

### Fine-tuning mental model

```text
Before training:
The model can summarize generally.

After training:
The model is nudged toward the target dataset style.
```

In [ ]:
# Check trainable parameters before training
params_before = count_parameters(model)
print("Trainable parameters before training:", f"{params_before['trainable']:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

loss_history = []
start_time = time.time()

model.train()

for step in range(1, TRAINING_STEPS + 1):
    batch = make_batch(tokenized_train, BATCH_SIZE)

    outputs = model(**batch)
    loss = outputs.loss

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    loss_value = float(loss.detach().cpu())
    loss_history.append(loss_value)

    if step == 1 or step % 5 == 0 or step == TRAINING_STEPS:
        print(f"Step {step:03d}/{TRAINING_STEPS} | loss = {loss_value:.4f}")

elapsed = time.time() - start_time
print("\nTraining completed in seconds:", round(elapsed, 2))
print("First loss:", round(loss_history[0], 4))
print("Last loss:", round(loss_history[-1], 4))

Trainable parameters before training: 76,961,152
Step 001/10 | loss = 3.9107
Step 005/10 | loss = 2.6934
Step 010/10 | loss = 1.6407

Training completed in seconds: 39.68
First loss: 3.9107
Last loss: 1.6407


# Part 12 — Evaluate after fine-tuning

Now compare:

```text
baseline summary
fine-tuned summary
human reference summary
```

A good live discussion:

- Did the fine-tuned model become more concise?
- Did it preserve meaning?
- Did it invent anything?
- Is it good enough for a manager handover?

In [ ]:
fine_tuned_summary = generate_summary(model, demo_row["dialogue"])

print("DIALOGUE PREVIEW:")
print(demo_row["dialogue"][:900])

print("\nBASELINE SUMMARY BEFORE FINE-TUNING:")
print(baseline_summary)

print("\nFINE-TUNED SUMMARY AFTER MINI TRAINING:")
print(fine_tuned_summary)

print("\nREFERENCE HUMAN SUMMARY:")
print(demo_row["summary"])

DIALOGUE PREVIEW:
#Person1#: Hello, how are you doing today?
#Person2#: I ' Ve been having trouble breathing lately.
#Person1#: Have you had any type of cold lately?
#Person2#: No, I haven ' t had a cold. I just have a heavy feeling in my chest when I try to breathe.
#Person1#: Do you have any allergies that you know of?
#Person2#: No, I don ' t have any allergies that I know of.
#Person1#: Does this happen all the time or mostly when you are active?
#Person2#: It happens a lot when I work out.
#Person1#: I am going to send you to a pulmonary specialist who can run tests on you for asthma.
#Person2#: Thank you for your help, doctor.

BASELINE SUMMARY BEFORE FINE-TUNING:
The doctor is going to send you to a pulmonary specialist who can run tests on you.

FINE-TUNED SUMMARY AFTER MINI TRAINING:
The doctor will send you to a pulmonary specialist who can run tests on you for asthma.

REFERENCE HUMAN SUMMARY:
#Person2# has trouble breathing. The doctor asks #Person2# about it and will send 

# Part 13 — Simple evaluation without extra packages

ROUGE is common for summarization evaluation, but it requires extra packages.

For classroom simplicity, we use a basic token-overlap score.

This is not a perfect evaluation metric.  
It is only a teaching-friendly signal.

Production evaluation should include:

- ROUGE / BERTScore,
- factuality checks,
- hallucination checks,
- human review,
- latency and cost.

In [ ]:
def clean_tokens(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    return [t for t in text.split() if len(t) > 2]

def overlap_score(reference: str, prediction: str) -> float:
    ref = set(clean_tokens(reference))
    pred = set(clean_tokens(prediction))
    if not ref:
        return 0.0
    return len(ref & pred) / len(ref)

def compare_summaries(row: Dict[str, Any]) -> Dict[str, Any]:
    before = generate_summary(model, row["dialogue"])
    reference = row["summary"]
    return {
        "reference": reference,
        "prediction": before,
        "overlap_score": round(overlap_score(reference, before), 3)
    }

# Evaluate only a few examples to keep runtime low.
eval_results = []
for row in val_rows[:3]:
    pred = generate_summary(model, row["dialogue"])
    eval_results.append({
        "topic": row.get("topic", ""),
        "reference": row["summary"],
        "prediction": pred,
        "overlap_score": round(overlap_score(row["summary"], pred), 3)
    })

for i, result in enumerate(eval_results, start=1):
    print(f"\n--- Evaluation example {i} ---")
    print("Topic:", result["topic"])
    print("Score:", result["overlap_score"])
    print("Prediction:", result["prediction"])
    print("Reference:", result["reference"])


--- Evaluation example 1 ---
Topic: experience in hotel
Score: 0.115
Prediction: The hotel offers a discount at the weekends.
Reference: #Person2# enjoys #Person2#'s weekend at the highland hotel because of the hotel's excellent and reasonably priced restaurant and good service. #Person2# introduces the hotel's facilities, weekend discount, and its interesting tip policy and suggests #Person1# make a reservation in advance.

--- Evaluation example 2 ---
Topic: in the studio
Score: 0.375
Prediction: Peter Wilson is a Green Peace activist. He works in London for the organization. They've been involved in anti-nuclear campaigns.
Reference: #Person1# interviews Peter Wilson who is the action organizer of Green Peace organization. #Person1# asks Peter to introduce to the audience what Green Peace is and what work it does. Peter also introduces detailed anti-nuclear campaigns.

--- Evaluation example 3 ---
Topic: recall something
Score: 0.222
Prediction: Mary was sitting in the cafeteria al

# Part 14 — Generalization test on a custom support dialogue

Now we test a dialogue that is not from DialogSum.

This connects the experiment back to OrbitCart and Module 23.

A model is useful only if it works beyond the exact training examples.

In [ ]:
custom_support_dialogue = '''
#Customer#: Hi, my laptop battery has become swollen and the bottom case is opening.
#Agent#: I am sorry to hear that. Is the device hot or leaking?
#Customer#: It is getting hot when I charge it.
#Agent#: Please stop charging it and keep it switched off.
#Customer#: Can I pack it and courier it back today?
#Agent#: I need to escalate this to the hazardous-device team. Please do not ship it until they review the case.
'''

custom_summary = generate_summary(model, custom_support_dialogue)

print("CUSTOM SUPPORT DIALOGUE:")
print(custom_support_dialogue)

print("\nMODEL SUMMARY:")
print(custom_summary)

CUSTOM SUPPORT DIALOGUE:

#Customer#: Hi, my laptop battery has become swollen and the bottom case is opening.
#Agent#: I am sorry to hear that. Is the device hot or leaking?
#Customer#: It is getting hot when I charge it.
#Agent#: Please stop charging it and keep it switched off.
#Customer#: Can I pack it and courier it back today?
#Agent#: I need to escalate this to the hazardous-device team. Please do not ship it until they review the case.


MODEL SUMMARY:
#Customer# has a battery problem and the bottom case is open. #Agent# will call the hazardous-device team to resolve it.


# Part 15 — What did full fine-tuning do?

Full fine-tuning updates all trainable parameters.

For FLAN‑T5-small this is manageable in a classroom mini-run.  
For large LLaMA/Mistral/Falcon-style models, full fine-tuning can become expensive.

| Approach | What changes | Cost | Classroom message |
|---|---|---:|---|
| Prompting only | no weights | low | quickest baseline |
| Full fine-tuning | all weights | high | strongest adaptation but costly |
| PEFT / LoRA | adapter weights | lower | common practical approach |
| RAG | external knowledge | medium | good for fresh/private knowledge |
| RLHF / DPO | preference alignment | medium/high | covered in Module 25 |

In [ ]:
params_after = count_parameters(model)

print("Parameter count after full fine-tuning:")
print(json.dumps(params_after, indent=2))

print("\nBecause we did full fine-tuning, trainable parameters are still the full model parameters.")

Parameter count after full fine-tuning:
{
  "total": 76961152,
  "trainable": 76961152
}

Because we did full fine-tuning, trainable parameters are still the full model parameters.


# Part 16 — LoRA / PEFT intuition

LoRA is a PEFT method.

Instead of updating every model weight, LoRA trains small adapter matrices.

### Simple analogy

```text
Full fine-tuning:
Retrain the whole employee.

LoRA:
Give the employee a small specialized operating manual.
```

### Why teams like LoRA

| Benefit | Explanation |
|---|---|
| Lower memory | fewer trainable parameters |
| Easier versioning | one base model + many adapters |
| Faster experiments | train small task-specific adapters |
| Better for limited GPUs | useful when full tuning is too costly |

The next cell is optional. It will not break the notebook if PEFT is unavailable.

In [ ]:
RUN_OPTIONAL_PEFT_LORA_DEMO = False

if RUN_OPTIONAL_PEFT_LORA_DEMO:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft>=0.13.0"])

    from peft import LoraConfig, TaskType, get_peft_model

    lora_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q", "v"]
    )

    lora_model = get_peft_model(lora_model, lora_config)
    lora_params = count_parameters(lora_model)

    print("LoRA model parameter count:")
    print(json.dumps(lora_params, indent=2))
    lora_model.print_trainable_parameters()

else:
    print("PEFT/LoRA demo is OFF for stable classroom execution.")
    print("Concept covered: LoRA trains small adapter weights instead of updating the whole model.")
    print("Turn RUN_OPTIONAL_PEFT_LORA_DEMO=True only if you want to test PEFT in your Colab runtime.")

PEFT/LoRA demo is OFF for stable classroom execution.
Concept covered: LoRA trains small adapter weights instead of updating the whole model.
Turn RUN_OPTIONAL_PEFT_LORA_DEMO=True only if you want to test PEFT in your Colab runtime.


# Part 17 — Quantization intuition

Fine-tuning changes behaviour.

Quantization changes deployment efficiency.

| Precision | Meaning | Typical use |
|---|---|---|
| FP32 | 32-bit floating point | training baseline |
| FP16/BF16 | 16-bit floating point | GPU inference/training |
| INT8 | 8-bit integer | smaller/faster inference |
| INT4 | 4-bit integer | very low-memory deployment, more risk |

A good teaching line:

```text
The more aggressively we compress, the more carefully we must evaluate.
```

# Part 18 — Post-training quantization vs QAT

| Method | When it happens | Simple meaning |
|---|---|---|
| Post-training quantization | after training | train first, compress later |
| Quantization-aware training | during training | train while simulating low precision |

For most beginner-friendly workflows:

```text
fine-tune → evaluate → quantize → evaluate again
```

That is the safest mental model.

# Part 19 — Safe INT8 dynamic quantization demo

This cell tries PyTorch dynamic quantization on CPU.

It is wrapped safely because not every transformer architecture/runtime combination supports every quantized generation path.

If quantized generation fails, the notebook will still continue and explain the result.

In [ ]:
def model_size_mb(model_obj, filename: str) -> float:
    path = PROJECT_DIR / filename
    torch.save(model_obj.state_dict(), path)
    return path.stat().st_size / (1024 * 1024)

fp32_size = model_size_mb(model.to("cpu"), "flan_t5_finetuned_fp32_state_dict.pt")
print("FP32-ish state_dict size MB:", round(fp32_size, 2))

# Keep future generation on CPU after this point.
device = "cpu"

try:
    from torch.ao.quantization import quantize_dynamic

    quantized_model = quantize_dynamic(
        model,
        {nn.Linear},
        dtype=torch.qint8
    )

    int8_size = model_size_mb(quantized_model, "flan_t5_dynamic_int8_state_dict.pt")
    print("INT8 dynamic state_dict size MB:", round(int8_size, 2))
    print("Size reduction:", round((1 - int8_size / fp32_size) * 100, 2), "%")

    try:
        q_summary = generate_summary(quantized_model, custom_support_dialogue)
        print("\nQuantized model summary:")
        print(q_summary)
    except Exception as generation_error:
        print("\nQuantized model was created, but generation failed in this runtime.")
        print("This can happen with some transformer modules and quantization backends.")
        print("Error:", type(generation_error).__name__, str(generation_error)[:300])

except Exception as quant_error:
    print("Dynamic quantization did not run in this runtime.")
    print("This is acceptable for class. The concept and deployment decision are still covered.")
    print("Error:", type(quant_error).__name__, str(quant_error)[:300])

FP32-ish state_dict size MB: 293.67


/tmp/ipykernel_964/1549477062.py:15: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = quantize_dynamic(


INT8 dynamic state_dict size MB: 120.73
Size reduction: 58.89 %

Quantized model summary:
Message from a customer urging them to buy refurbished laptops and chargers.


# Part 20 — Accuracy vs size vs latency decision

After quantization, always evaluate again.

| Question | Why it matters |
|---|---|
| Did summaries become shorter but less useful? | quality risk |
| Did the model invent facts? | hallucination risk |
| Did safety behaviour degrade? | deployment risk |
| Is latency meaningfully better? | business value |
| Is size reduction worth the quality trade-off? | deployment decision |

A compressed model is not automatically a better model.

In [ ]:
decision_table = [
    {
        "deployment_need": "Highest quality",
        "recommended_strategy": "Use full-size or lightly optimized model",
        "risk": "Higher cost/latency"
    },
    {
        "deployment_need": "Low cost and acceptable quality",
        "recommended_strategy": "Try INT8 after evaluation",
        "risk": "Small quality drop possible"
    },
    {
        "deployment_need": "Edge or very low memory",
        "recommended_strategy": "Consider INT4 / small model / PEFT + quantization",
        "risk": "Higher quality and safety risk"
    },
    {
        "deployment_need": "Private/fresh knowledge",
        "recommended_strategy": "Use RAG, possibly with fine-tuned model",
        "risk": "Retrieval quality matters"
    },
    {
        "deployment_need": "Better human preference alignment",
        "recommended_strategy": "Collect preference data for Module 25 DPO/RLHF",
        "risk": "Feedback bias and review cost"
    }
]

print(json.dumps(decision_table, indent=2))

[
  {
    "deployment_need": "Highest quality",
    "recommended_strategy": "Use full-size or lightly optimized model",
    "risk": "Higher cost/latency"
  },
  {
    "deployment_need": "Low cost and acceptable quality",
    "recommended_strategy": "Try INT8 after evaluation",
    "risk": "Small quality drop possible"
  },
  {
    "deployment_need": "Edge or very low memory",
    "recommended_strategy": "Consider INT4 / small model / PEFT + quantization",
    "risk": "Higher quality and safety risk"
  },
  {
    "deployment_need": "Private/fresh knowledge",
    "recommended_strategy": "Use RAG, possibly with fine-tuned model",
    "risk": "Retrieval quality matters"
  },
  {
    "deployment_need": "Better human preference alignment",
    "recommended_strategy": "Collect preference data for Module 25 DPO/RLHF",
    "risk": "Feedback bias and review cost"
  }
]


# Part 21 — Production tools map

This experiment used a simple Hugging Face workflow.

In real production, teams may add:

| Tool | Role |
|---|---|
| Hugging Face Transformers | load/train/infer models |
| Hugging Face Datasets | dataset loading and preprocessing |
| PEFT | LoRA/adapters |
| Accelerate | GPU/distributed training support |
| bitsandbytes | 8-bit/4-bit loading/training on supported GPUs |
| ONNX Runtime | optimized inference |
| Intel Neural Compressor | compression and quantization workflows |
| OpenVINO | deployment on Intel CPU/GPU/NPU |
| Evaluation tools | quality/safety/latency regression checks |

Do not introduce all of these at once to beginners.

Teach the workflow first:

```text
data → baseline → fine-tune → evaluate → optimize → evaluate again
```

# Part 22 — LLaMA / Mistral / Falcon roadmap

This notebook uses FLAN‑T5-small because it is manageable for class.

For larger decoder-style models, the production workflow is similar but heavier:

```text
choose base model
→ choose prompt/chat format
→ prepare instruction dataset
→ train full model or LoRA adapter
→ evaluate
→ quantize or serve in lower precision
→ evaluate again
→ deploy and monitor
```

For LLaMA/Mistral-style models, you commonly see:

| Topic | Production direction |
|---|---|
| Full fine-tuning | expensive, needs strong GPU setup |
| LoRA / QLoRA | common practical route |
| 8-bit / 4-bit | often used for memory efficiency |
| RAG + fine-tuning | common enterprise pattern |
| DPO/RLHF | preference alignment after SFT |

# Part 23 — Safe and ethical deployment

For a summarizer, safety is not only about offensive content.

A summarizer can fail by:

- inventing an action that never happened,
- dropping an important risk,
- exposing private information,
- changing who said what,
- making a confident summary from unclear dialogue.

### Classroom discussion

Ask students:

```text
Would you send this summary directly to a customer?
Would you send it only to a support manager?
Should a human approve it?
What data should never be used for fine-tuning?
```

In [ ]:
def simple_summary_safety_check(dialogue: str, summary: str) -> Dict[str, Any]:
    issues = []

    if len(summary.strip()) < 20:
        issues.append("Summary may be too short to be useful.")

    if "refund" in summary.lower() and "refund" not in dialogue.lower():
        issues.append("Summary mentions refund but dialogue may not support it.")

    risky_terms = ["swollen", "leaking", "smoke", "overheating", "unauthorized", "password"]
    if any(term in dialogue.lower() for term in risky_terms):
        if not any(term in summary.lower() for term in risky_terms):
            issues.append("Summary may have dropped an important risk signal.")

    return {
        "pass": len(issues) == 0,
        "issues": issues
    }

safety_result = simple_summary_safety_check(custom_support_dialogue, custom_summary)

print("Custom summary:")
print(custom_summary)
print("\nSafety check:")
print(json.dumps(safety_result, indent=2))

Custom summary:
#Customer# has a battery problem and the bottom case is open. #Agent# will call the hazardous-device team to resolve it.

Safety check:
{
  "pass": false,
  "issues": [
    "Summary may have dropped an important risk signal."
  ]
}


# Part 25 — Final student challenge

Give students a new dialogue and ask them to prepare a deployment recommendation.

They should answer:

1. Is prompt-only enough?
2. Would SFT help?
3. Would LoRA be better than full fine-tuning?
4. What evaluation would you run?
5. Would you quantize?
6. What safety risk must be checked?
7. How would this connect to Module 25 feedback learning?

### Final teaching line

```text
Fine-tuning is not only training code.
It is a full decision pipeline:
data quality → model adaptation → evaluation → optimization → deployment safety.
```